# 00_03: HuggingFace Ecosystem Tour

**Learning Objectives:**
- Navigate the HuggingFace Hub to find models, datasets, and Spaces
- Use `pipeline()` for quick inference on any task
- Understand AutoClasses (`AutoModel`, `AutoTokenizer`, `AutoConfig`) and when to use each
- Read model cards and understand model metadata
- Use the `HfApi` to programmatically interact with the Hub

**Prerequisites:** Notebooks 00_01 and 00_02. Internet connection required for Hub access.

## Prerequisites

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum |
| **Storage** | ~1GB for model downloads |
| **GPU** | Not required |
| **Internet** | Required for Hub access |

## Expected Behaviors

### What You'll See
- `pipeline()` makes any HuggingFace task a one-liner
- AutoClasses automatically select the right model architecture from a model name
- The Hub has 500k+ models, 100k+ datasets, and 300k+ Spaces
- Model cards contain critical metadata about model training, biases, and intended use

### First Run
- Several small model downloads (~500MB total)
- Hub API calls require internet but no authentication for public models

### Common Observations
- `pipeline()` handles tokenization, inference, and post-processing automatically
- The same model name works with `AutoModel`, `AutoTokenizer`, and `AutoConfig`
- Some models require authentication (gated models like Llama)

## Overview

The HuggingFace ecosystem is the largest open-source ML platform, built around three pillars:

1. **Hub** — A GitHub for ML: models, datasets, and demo apps (Spaces)
2. **Transformers** — The Python library that loads and runs models from the Hub
3. **Supporting Libraries** — `datasets`, `tokenizers`, `accelerate`, `diffusers`, `gradio`, and more

This notebook is your orientation tour. After completing it, you'll know how to find any model, load it, run inference, and understand its documentation — which is everything you need for the rest of this tutorial series.

### The Auto-Everything Philosophy

HuggingFace's key design principle is **Auto-dispatch**: you provide a model name (e.g., `"bert-base-uncased"`), and the library automatically selects the right tokenizer class, model class, and configuration. You rarely need to import specific model classes.

## Setup and Installation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import random
import transformers
from transformers import (
    pipeline,
    AutoModel,
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
)
from huggingface_hub import HfApi, ModelFilter
import pandas as pd
import sys

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Part 1: The Pipeline API — One-Line Inference

The `pipeline()` function is the fastest way to use any HuggingFace model. It wraps tokenization, model inference, and post-processing into a single callable object. You specify a **task** (e.g., `"sentiment-analysis"`) and optionally a **model name** — if you don't specify a model, it picks a sensible default.

This is the recommended starting point for any new task. Once you understand your task, you can switch to manual model loading for more control.

In [ ]:
# Sentiment Analysis — the "Hello World" of NLP
classifier = pipeline('sentiment-analysis', device=device)

results = classifier([
    'I love learning about transformers!',
    'This is really confusing and frustrating.',
    'The weather is okay today.',
])

for text, result in zip(
    ['I love learning...', 'This is confusing...', 'Weather is okay...'],
    results
):
    print(f'{text:30s} → {result["label"]:8s} ({result["score"]:.4f})')

Notice we didn't specify a model — `pipeline` defaulted to `distilbert-base-uncased-finetuned-sst-2-english`, a fine-tuned DistilBERT model. Let's explore more pipeline tasks to see the breadth of what's available.

In [ ]:
# Text Generation
generator = pipeline('text-generation', model='distilgpt2', device=device)
result = generator(
    'Artificial intelligence will',
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
)
print('=== Text Generation ===')
print(result[0]['generated_text'])

In [ ]:
# Named Entity Recognition
ner = pipeline('ner', aggregation_strategy='simple', device=device)
entities = ner('Hugging Face is a company based in New York City founded by Clément Delangue.')

print('=== Named Entity Recognition ===')
ner_df = pd.DataFrame([{
    'Entity': e['word'],
    'Type': e['entity_group'],
    'Score': f'{e["score"]:.4f}',
} for e in entities])
ner_df

In [ ]:
# Question Answering (Extractive)
qa = pipeline('question-answering', device=device)
result = qa(
    question='What is the capital of France?',
    context='France is a country in Western Europe. Its capital city is Paris, '
            'which is also the largest city in the country.'
)
print('=== Question Answering ===')
print(f'Answer: {result["answer"]}')
print(f'Score: {result["score"]:.4f}')
print(f'Position: chars {result["start"]}-{result["end"]}')

In [ ]:
# Summarization
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6', device=device)

long_text = (
    'The Transformer architecture was introduced in 2017 in the landmark paper '
    '"Attention Is All You Need" by Vaswani et al. Unlike previous sequence models '
    'like RNNs and LSTMs, Transformers process all positions in parallel using '
    'self-attention mechanisms. This architectural innovation led to dramatic '
    'improvements in training speed and model quality. Since then, Transformers '
    'have become the foundation of virtually all state-of-the-art NLP models, '
    'including BERT, GPT, T5, and their many variants. The architecture has also '
    'been successfully applied to computer vision, audio processing, and '
    'multimodal tasks.'
)

summary = summarizer(long_text, max_length=50, min_length=20)
print('=== Summarization ===')
print(f'Original: {len(long_text.split())} words')
print(f'Summary:  {summary[0]["summary_text"]}')

### Available Pipeline Tasks

The `pipeline()` function supports dozens of tasks. Here are the most common ones you'll encounter in this tutorial:

| Task | Pipeline String | Section |
|------|----------------|--------|
| Sentiment Analysis | `"sentiment-analysis"` | 01 NLP |
| Text Generation | `"text-generation"` | 01 NLP |
| Summarization | `"summarization"` | 01 NLP |
| NER | `"ner"` | 01 NLP |
| Question Answering | `"question-answering"` | 01 NLP |
| Translation | `"translation"` | 01 NLP |
| Image Classification | `"image-classification"` | 02 CV |
| Object Detection | `"object-detection"` | 02 CV |
| Image Segmentation | `"image-segmentation"` | 02 CV |
| Speech Recognition | `"automatic-speech-recognition"` | 03 Audio |
| Text-to-Speech | `"text-to-speech"` | 03 Audio |
| Image-to-Text | `"image-to-text"` | 04 Multimodal |
| Visual QA | `"visual-question-answering"` | 04 Multimodal |

## Part 2: AutoClasses — The Universal Loader

While `pipeline()` is convenient, sometimes you need more control — custom decoding strategies, access to hidden states, or specific model configurations. That's where **AutoClasses** come in.

The key insight: you almost never need to import model-specific classes like `BertModel` or `GPT2Tokenizer`. The `Auto` versions inspect the model's `config.json` on the Hub and automatically dispatch to the right class.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'

# AutoConfig — loads model configuration (hyperparameters, architecture)
config = AutoConfig.from_pretrained(MODEL_NAME)
print('=== AutoConfig ===')
print(f'Architecture: {config.architectures}')
print(f'Hidden size: {config.hidden_size}')
print(f'Layers: {config.n_layers}')
print(f'Attention heads: {config.n_heads}')
print(f'Vocab size: {config.vocab_size}')
print(f'Max position: {config.max_position_embeddings}')

In [ ]:
# AutoTokenizer — loads the correct tokenizer for any model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('=== AutoTokenizer ===')
print(f'Type: {type(tokenizer).__name__}')
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Model max length: {tokenizer.model_max_length}')
print(f'Special tokens: {tokenizer.special_tokens_map}')

# Tokenize a sample
text = 'HuggingFace makes NLP accessible to everyone!'
tokens = tokenizer(text, return_tensors='pt')
print(f'\nInput: "{text}"')
print(f'Token IDs: {tokens["input_ids"][0].tolist()}')
print(f'Tokens: {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')
print(f'Attention mask: {tokens["attention_mask"][0].tolist()}')

In [ ]:
# AutoModel — loads the base model (without a task-specific head)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print('=== AutoModel ===')
print(f'Type: {type(model).__name__}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Forward pass
with torch.no_grad():
    outputs = model(**tokens.to(device))

print(f'\nOutput type: {type(outputs).__name__}')
print(f'Last hidden state shape: {outputs.last_hidden_state.shape}')
print('  → (batch_size, sequence_length, hidden_size)')

### AutoModel vs Task-Specific AutoModels

`AutoModel` gives you the base Transformer without any task-specific output layer. For specific tasks, use the task-specific variant:

| Class | Task | Output |
|-------|------|--------|
| `AutoModel` | None (base) | Hidden states |
| `AutoModelForSequenceClassification` | Classification | Logits per class |
| `AutoModelForTokenClassification` | NER | Logits per token |
| `AutoModelForCausalLM` | Text generation | Next-token logits |
| `AutoModelForSeq2SeqLM` | Translation/Summarization | Decoder logits |
| `AutoModelForQuestionAnswering` | Extractive QA | Start/end logits |

Let's see the difference in practice.

In [ ]:
def compare_model_outputs(
    model_name: str,
    text: str,
) -> pd.DataFrame:
    """Compare outputs of base model vs classification model.

    Args:
        model_name: HuggingFace model identifier (must have a classification variant).
        text: Input text.

    Returns:
        DataFrame comparing the two model types.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    inputs = tokenizer(text, return_tensors='pt').to(device)
    
    # Base model
    base_model = AutoModel.from_pretrained(model_name).to(device)
    base_model.eval()
    with torch.no_grad():
        base_out = base_model(**inputs)
    
    # Classification model
    cls_model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
    cls_model = AutoModelForSequenceClassification.from_pretrained(
        cls_model_name
    ).to(device)
    cls_model.eval()
    cls_tokenizer = AutoTokenizer.from_pretrained(cls_model_name)
    cls_inputs = cls_tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        cls_out = cls_model(**cls_inputs)
    
    probs = torch.softmax(cls_out.logits, dim=-1)[0]
    
    return pd.DataFrame({
        'Property': [
            'Model class', 'Output type', 'Output shape',
            'Use case', 'Result',
        ],
        'AutoModel (base)': [
            type(base_model).__name__,
            'BaseModelOutput',
            str(tuple(base_out.last_hidden_state.shape)),
            'Feature extraction, custom heads',
            f'768-dim vector per token',
        ],
        'AutoModelForSequenceClassification': [
            type(cls_model).__name__,
            'SequenceClassifierOutput',
            str(tuple(cls_out.logits.shape)),
            'Text classification',
            f'POSITIVE: {probs[1]:.4f}, NEGATIVE: {probs[0]:.4f}',
        ],
    })


compare_model_outputs(MODEL_NAME, 'I really enjoy using HuggingFace!')

## Part 3: Navigating the HuggingFace Hub

The HuggingFace Hub hosts over 500,000 models, 100,000 datasets, and 300,000 Spaces (demo apps). Let's learn how to search and filter programmatically using the `HfApi`.

In [ ]:
api = HfApi()


def search_models(
    task: str | None = None,
    search_query: str | None = None,
    limit: int = 5,
) -> pd.DataFrame:
    """Search the HuggingFace Hub for models.

    Args:
        task: Pipeline task filter (e.g., 'text-classification').
        search_query: Text search query.
        limit: Maximum number of results.

    Returns:
        DataFrame with model search results.
    """
    models = api.list_models(
        filter=task,
        search=search_query,
        sort='downloads',
        direction=-1,
        limit=limit,
    )
    
    data: list[dict[str, object]] = []
    for model in models:
        data.append({
            'Model ID': model.modelId,
            'Downloads (30d)': f'{model.downloads:,}' if model.downloads else 'N/A',
            'Likes': model.likes,
            'Pipeline Tag': model.pipeline_tag or 'N/A',
        })
    
    return pd.DataFrame(data)


print('=== Most Downloaded Text Classification Models ===')
search_models(task='text-classification', limit=5)

In [ ]:
print('=== Most Downloaded Text Generation Models ===')
search_models(task='text-generation', limit=5)

In [ ]:
print('=== Search for "whisper" models ===')
search_models(search_query='whisper', limit=5)

The Hub search is powerful for discovery. You can filter by:
- **Task**: `text-classification`, `text-generation`, `image-classification`, etc.
- **Library**: `pytorch`, `tensorflow`, `jax`, etc.
- **Language**: `en`, `fr`, `zh`, etc.
- **Dataset**: Models trained on specific datasets
- **License**: `apache-2.0`, `mit`, etc.

When browsing on [huggingface.co](https://huggingface.co/models), use the left sidebar filters for the same functionality.

## Part 4: Understanding Model Cards

Every model on the Hub should have a **model card** — documentation that explains what the model does, how it was trained, its limitations, and its intended use. Model cards are critical for responsible AI usage. Let's inspect one programmatically.

In [ ]:
def inspect_model_card(model_id: str) -> None:
    """Print key information from a model's Hub metadata.

    Args:
        model_id: HuggingFace model identifier.
    """
    model_info = api.model_info(model_id)
    
    print(f'=== Model Card: {model_id} ===')
    print(f'Author: {model_info.author}')
    print(f'Pipeline tag: {model_info.pipeline_tag}')
    print(f'Library: {model_info.library_name}')
    print(f'Downloads (30d): {model_info.downloads:,}')
    print(f'Likes: {model_info.likes}')
    print(f'License: {model_info.card_data.license if model_info.card_data else "N/A"}')
    
    if model_info.tags:
        print(f'Tags: {", ".join(model_info.tags[:10])}')
    
    # List files in the repository
    siblings = model_info.siblings or []
    files = [s.rfilename for s in siblings]
    print(f'\nFiles in repo ({len(files)} total):')
    for f in files[:10]:
        print(f'  {f}')
    if len(files) > 10:
        print(f'  ... and {len(files) - 10} more')


inspect_model_card('distilbert-base-uncased')

### Key Files in a Model Repository

When you look at a model on the Hub, you'll see these standard files:

| File | Purpose |
|------|---------|
| `config.json` | Model architecture and hyperparameters |
| `model.safetensors` or `pytorch_model.bin` | Model weights |
| `tokenizer.json` | Fast tokenizer data |
| `tokenizer_config.json` | Tokenizer configuration |
| `vocab.txt` or `merges.txt` | Vocabulary files |
| `special_tokens_map.json` | Special token definitions |
| `README.md` | The model card (documentation) |

The `config.json` is what AutoClasses use to determine which model class to load.

In [ ]:
# Look at the raw config to understand auto-dispatch
config = AutoConfig.from_pretrained('distilbert-base-uncased')
print('Config determines the model class:')
print(f'  model_type: {config.model_type}')
print(f'  architectures: {config.architectures}')
print(f'\nWhen you call AutoModel.from_pretrained("distilbert-base-uncased"),')
print(f'it reads config.json, sees model_type="{config.model_type}",'
      f' and imports {config.architectures[0]}.')

## Part 5: The Datasets Library (Preview)

The `datasets` library provides access to 100,000+ datasets on the Hub, with efficient streaming, caching, and processing. We'll cover it in depth in notebook 05_04, but here's a quick preview to show how it fits into the ecosystem.

In [ ]:
from datasets import load_dataset

# Load a small sample of a popular dataset
dataset = load_dataset('imdb', split='test[:5]')

print('=== IMDB Dataset Sample ===')
print(f'Features: {list(dataset.features.keys())}')
print(f'Number of examples: {len(dataset)}')
print(f'\nFirst example:')
print(f'  Text (truncated): {dataset[0]["text"][:150]}...')
print(f'  Label: {dataset[0]["label"]} ({"positive" if dataset[0]["label"] == 1 else "negative"})')

In [ ]:
# Search for datasets on the Hub
datasets_list = api.list_datasets(
    search='sentiment',
    sort='downloads',
    direction=-1,
    limit=5,
)

print('=== Top Sentiment Datasets ===')
ds_data: list[dict[str, object]] = []
for ds in datasets_list:
    ds_data.append({
        'Dataset ID': ds.id,
        'Downloads': f'{ds.downloads:,}' if ds.downloads else 'N/A',
        'Likes': ds.likes,
    })
pd.DataFrame(ds_data)

## Part 6: Putting It All Together

Now let's combine everything we've learned into a practical workflow. This demonstrates the typical pattern you'll use throughout this tutorial series:

1. **Discover** a model on the Hub
2. **Load** it with AutoClasses
3. **Run inference** with pipeline or manual loading
4. **Interpret** the results

In [ ]:
def full_workflow_demo(
    model_name: str,
    task: str,
    texts: list[str],
) -> pd.DataFrame:
    """Demonstrate the full HuggingFace workflow for a given task.

    Args:
        model_name: HuggingFace model identifier.
        task: Pipeline task string.
        texts: List of input texts.

    Returns:
        DataFrame with results for each input.
    """
    # Step 1: Inspect model metadata
    config = AutoConfig.from_pretrained(model_name)
    print(f'Model: {model_name}')
    print(f'Architecture: {config.architectures}')
    print(f'Task: {task}')
    
    # Step 2: Create pipeline
    pipe = pipeline(task, model=model_name, device=device)
    
    # Step 3: Run inference
    results = pipe(texts)
    
    # Step 4: Format results
    data: list[dict[str, str]] = []
    for text, result in zip(texts, results):
        truncated = text[:50] + '...' if len(text) > 50 else text
        if isinstance(result, dict):
            data.append({
                'Input': truncated,
                'Label': result.get('label', 'N/A'),
                'Score': f'{result.get("score", 0):.4f}',
            })
        elif isinstance(result, list):
            top = result[0] if result else {}
            data.append({
                'Input': truncated,
                'Label': top.get('label', str(top.get('entity_group', 'N/A'))),
                'Score': f'{top.get("score", 0):.4f}',
            })
    
    return pd.DataFrame(data)


# Demo: Full sentiment analysis workflow
full_workflow_demo(
    model_name='distilbert-base-uncased-finetuned-sst-2-english',
    task='sentiment-analysis',
    texts=[
        'HuggingFace has revolutionized the ML ecosystem!',
        'This model is terrible at understanding context.',
        'I guess transformers are becoming pretty standard now.',
        'The documentation could really use some improvement.',
    ]
)

## Part 7: Saving and Loading Models Locally

You can save models and tokenizers to disk for offline use, faster loading, or sharing. This is especially useful when you've fine-tuned a model (covered in notebooks 01_07 and 01_08).

In [ ]:
import os
import tempfile

# Save model and tokenizer to a temporary directory
save_dir = os.path.join(tempfile.gettempdir(), 'hf_tutorial_model')

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModel.from_pretrained('distilbert-base-uncased')

tokenizer.save_pretrained(save_dir)
model.save_pretrained(save_dir)

print(f'Saved to: {save_dir}')
print(f'\nSaved files:')
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    if size > 1_000_000:
        print(f'  {f:40s} {size / 1_000_000:.1f} MB')
    else:
        print(f'  {f:40s} {size / 1_000:.1f} KB')

In [ ]:
# Load from the saved directory (works offline!)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)
loaded_model = AutoModel.from_pretrained(save_dir).to(device)
loaded_model.eval()

# Verify it works
test_input = loaded_tokenizer('Hello from a locally saved model!', return_tensors='pt').to(device)
with torch.no_grad():
    test_output = loaded_model(**test_input)

print(f'Loaded model type: {type(loaded_model).__name__}')
print(f'Output shape: {test_output.last_hidden_state.shape}')
print('Model loaded and working from local directory!')

# Cleanup
import shutil
shutil.rmtree(save_dir, ignore_errors=True)
print(f'\nCleaned up temporary directory.')

## HuggingFace Ecosystem Map

Here's how all the HuggingFace libraries fit together. This tutorial primarily uses `transformers` and `datasets`, but you'll encounter others:

| Library | Purpose | Used In |
|---------|---------|--------|
| `transformers` | Load and run models | Every notebook |
| `datasets` | Load and process datasets | 05_04, fine-tuning |
| `tokenizers` | Fast tokenization (Rust backend) | Under the hood of `transformers` |
| `accelerate` | Multi-GPU, mixed precision | Fine-tuning notebooks |
| `peft` | LoRA and other efficient fine-tuning | 01_08 |
| `diffusers` | Stable Diffusion and image generation | 04_03, 04_04 |
| `gradio` | Interactive demo apps | 05_05 |
| `huggingface_hub` | Python API for the Hub | This notebook |
| `safetensors` | Safe, fast model serialization | Under the hood |
| `evaluate` | Standardized evaluation metrics | Fine-tuning notebooks |
| `optimum` | ONNX export and optimization | 05_06 |

## Performance Benchmarking

Let's compare the two main approaches — `pipeline()` vs manual model loading — to understand the tradeoff between convenience and control.

In [ ]:
import time


def benchmark_approaches(
    model_name: str,
    texts: list[str],
    num_runs: int = 5,
) -> pd.DataFrame:
    """Compare pipeline vs manual model loading performance.

    Args:
        model_name: HuggingFace model identifier for classification.
        texts: List of input texts.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame comparing the two approaches.
    """
    # Approach 1: Pipeline
    pipe = pipeline('sentiment-analysis', model=model_name, device=device)
    pipe_times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        pipe(texts)
        pipe_times.append(time.perf_counter() - start)
    
    # Approach 2: Manual
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()
    
    manual_times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        manual_times.append(time.perf_counter() - start)
    
    return pd.DataFrame({
        'Approach': ['pipeline()', 'Manual (AutoModel + tokenizer)'],
        'Mean Latency (ms)': [
            f'{np.mean(pipe_times) * 1000:.1f}',
            f'{np.mean(manual_times) * 1000:.1f}',
        ],
        'Lines of Code': ['1', '~5'],
        'Control Level': ['Low (automatic)', 'High (full access)'],
        'Best For': ['Quick experiments', 'Production / custom logic'],
    })


test_texts = [
    'I love machine learning!',
    'This is terrible.',
    'Not sure how I feel about this.',
]
benchmark_approaches(
    'distilbert-base-uncased-finetuned-sst-2-english',
    test_texts,
)

## Exercises

1. **Explore the Hub**: Use `search_models()` to find the most downloaded model for `image-classification`. Load it with `pipeline()` and classify an image from a URL.

2. **Compare tokenizers**: Load tokenizers for `bert-base-uncased`, `gpt2`, and `t5-small`. Tokenize the same sentence with each and compare the outputs (vocabulary size, number of tokens, special tokens).

3. **Model card analysis**: Use `inspect_model_card()` on 3 different models. Which has the most thorough documentation? What information is missing?

4. **Zero-shot classification**: Try `pipeline('zero-shot-classification')` — this lets you classify text into arbitrary categories without training. Test it with custom labels.

## Key Takeaways

- **`pipeline()`** is the fastest way to use any HuggingFace model — one line for tokenization, inference, and post-processing
- **AutoClasses** (`AutoModel`, `AutoTokenizer`, `AutoConfig`) automatically dispatch to the correct model class based on `config.json`
- The **HuggingFace Hub** hosts 500k+ models, searchable by task, downloads, and metadata via `HfApi`
- **Model cards** are essential documentation — always check a model's intended use, limitations, and training data before deploying it
- Use `pipeline()` for quick experiments and `AutoModel` + `AutoTokenizer` for production code that needs fine-grained control

## Next Steps & Resources

**Next section**: [01_01 — Text Generation](../01_nlp/01_01_nlp_text_generation.ipynb) — apply what you've learned to build a text generation system.

**Resources:**
- [HuggingFace Hub](https://huggingface.co/models) — Browse and search models
- [Transformers Documentation](https://huggingface.co/docs/transformers/) — Complete API reference
- [Pipeline Tutorial](https://huggingface.co/docs/transformers/pipeline_tutorial) — Official pipeline guide
- [AutoClass Tutorial](https://huggingface.co/docs/transformers/autoclass_tutorial) — How auto-dispatch works
- [Hub Documentation](https://huggingface.co/docs/hub/) — Hub features and API
- [Model Card Guide](https://huggingface.co/docs/hub/model-cards) — How to read and write model cards